# Profiling Bronze

Este notebook perfila la capa Bronze. Bronze conserva los datos aterrizados en Parquet, con trazabilidad de ingesta y sin modelado dimensional.

Objetivo:
- Inventariar tablas Bronze.
- Revisar registros y columnas.
- Medir nulos por columna.
- Mostrar muestras para evidencia.
- Generar reportes HTML separados en `reports/profiling/bronze`.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

ROOT = Path('/home/jovyan/work')
BRONZE = ROOT / 'data' / 'bronze'
spark = SparkSession.builder.appName('bronze-profiling-notebook').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
tables = sorted(p for p in BRONZE.iterdir() if p.is_dir() and any(p.rglob('*.parquet')))
[(p.name, str(p)) for p in tables]

In [ ]:
summary = []
for path in tables:
    df = spark.read.parquet(str(path))
    summary.append((path.name, df.count(), len(df.columns)))
spark.createDataFrame(summary, ['tabla_bronze', 'registros', 'columnas']).orderBy('tabla_bronze').show(200, truncate=False)

## Calidad Basica Bronze

Bronze valida disponibilidad, lectura, trazabilidad y completitud inicial. Los problemas detectados no se corrigen aqui; se documentan para tratamiento posterior en Silver.

In [ ]:
for path in tables:
    df = spark.read.parquet(str(path))
    print('\n===', path.name, '===')
    df.printSchema()
    cols = df.columns[:25]
    if cols:
        nulls = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in cols])
        nulls.show(truncate=False)
    df.limit(5).show(truncate=False)

In [ ]:
# Generar HTML de profiling para Bronze, Silver y Gold.
# La salida Bronze queda en reports/profiling/bronze/index.html
%run ../../scripts/profile_medallion_layers.py